# Lab 56: Production traces and routing

Run the eval/cost loop on the spans the agent emits, not on hand-built data: instrument steps as OpenTelemetry GenAI spans, reconstruct them from the exported spans, and compute cost over them. Then replace Lab 53's hand-set `simple` flag with a learned routing classifier that has its own eval. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
from traces import (make_tracer, instrument, steps_from_spans, token_cost,
                    routing_eval, _synthetic_sessions, GENAI_OP)
sessions = _synthetic_sessions()
# Labs 51 and 53 ran on hand-built data. Here the cost loop runs on the spans the agent emits,
# and the routing flag is learned, not hand-set.
print(f"{len(sessions)} sessions; instrumenting each step as a GenAI span")

## Step 1: Instrument steps as GenAI spans

In [ ]:
# Emit one OpenTelemetry span per step with the gen_ai.* semantic-convention attributes a real
# instrumentation sets (system, model, input/output tokens), plus the agent fields the loops need.
tracer, exporter = make_tracer()
instrument(tracer, sessions)
spans = exporter.get_finished_spans()
print(f"exported {len(spans)} spans named '{GENAI_OP}'")
a = spans[0].attributes
for k in ["gen_ai.system","gen_ai.request.model","gen_ai.usage.input_tokens","gen_ai.usage.output_tokens","agent.session_id"]:
    print(f"  {k} = {a[k]}")

## Step 2: Run the cost loop on the exported spans

In [ ]:
# TODO: reconstruct steps from the exported spans with steps_from_spans(exporter) and compute
# token_cost over them. Compare to the cost computed from the source sessions. Why does it matter
# that they match?
raise NotImplementedError

## Step 3: A learned routing classifier with its own eval

In [ ]:
# TODO: call routing_eval(sessions). Report the classifier precision/recall and compare the
# learned routing saving to the oracle saving. What does the gap represent, and why does the
# routing decision need its own eval?
raise NotImplementedError

## What you built

The eval/cost loop on real traces, and a learned routing decision. **Instrumentation**: each agent step is emitted as an OpenTelemetry span following the GenAI semantic conventions (`gen_ai.system`, `gen_ai.request.model`, `gen_ai.usage.input_tokens` / `output_tokens`), exported through an in-memory exporter. **Decoupled loop**: the cost computation reads the *exported spans* and matches the source exactly - so in production the loop runs on the trace store the agent already writes to, not on a parallel data structure. **Learned routing**: Lab 53's hand-set `simple` flag is replaced by a logistic-regression classifier on step features, with a held-out precision/recall eval; routing on its predictions recovers most of the oracle saving (here ~25% of a 33% ceiling), and the gap is the cost of misroutes.

**Where this simplifies:** the spans are exported in-process to an `InMemorySpanExporter` so the lab is offline and deterministic - production points the same instrumentation at an OTLP collector and the loops query the trace backend, but the span shape and the reconstruction are identical. The classifier is a small logistic regression on four features; a real router uses richer features and is retrained as traffic shifts. The headline lesson holds regardless: **a learned router needs its own eval**, because a misroute either spends too much (a hard step sent to a cheap model retries) or risks quality (routing a step that needed the strong model) - so you measure the routed path, you don't trust the route.